# SHQ Analysis Program 2 - Single Leg Drop Landing Knee Flexion
Updated 04/20/2026

-----
## Cell 1 - Imports
Import the necessary packages to run the program.

In [2]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from tkinter import Tk
from tkinter.filedialog import askdirectory, asksaveasfilename
from IPython.display import display

print(f'{datetime.now()} - Imports OK')

custom_theme = {
    "axes.spines.right":  False,
    "axes.spines.top":    False,
    "axes.titlelocation": "left",
    "axes.titley":        1,
    "font.weight":        "bold",
    "axes.titlesize":     "x-large",
    "axes.labelsize":     "x-large",
    "axes.titleweight":   "bold",
    "axes.labelweight":   "bold"
}

plt.rcParams.update({
    **custom_theme,
    'figure.facecolor': 'white',
    'axes.labelcolor':  'black',
    'axes.edgecolor':   'black',
    'xtick.color':      'black',
    'ytick.color':      'black',
    'axes.titlecolor':  'black'
})

2026-04-20 21:16:25.415251 - Imports OK


-----
## Cell 2 - Define Functions

`pick_directory` — opens a folder browser dialog.

`pick_save_path` — opens a Save As dialog.

`parse_leg_from_trial` — extracts the leg (Left/Right) from a trial name like `L_drop_1` or `R_drop_3`.

`get_knee_angle_col` — returns the correct knee angle column name for a given leg.

In [3]:
def pick_directory(prompt="Select Folder"):
    root = Tk()
    root.attributes('-topmost', True)
    root.after(1, root.attributes, '-topmost', False)
    folder = askdirectory(title=prompt)
    root.destroy()
    return folder


def pick_save_path(default_name):
    root = Tk()
    root.attributes('-topmost', True)
    root.after(1, root.attributes, '-topmost', False)
    path = asksaveasfilename(
        title="Choose where to save the drop landing results",
        initialfile=default_name,
        defaultextension=".xlsx",
        filetypes=[("Excel files", "*.xlsx")]
    )
    root.destroy()
    return path


def parse_leg_from_trial(trial_name):
    """
    Extract leg from trial name.
    'L_drop_1' -> 'Left'
    'R_drop_2' -> 'Right'
    """
    match = re.match(r'([LR])_drop_(\d+)', trial_name, re.IGNORECASE)
    if match:
        return 'Left' if match.group(1).upper() == 'L' else 'Right'
    return None


def get_knee_angle_col(leg):
    """
    Return the correct CSV column name for the given leg.
    """
    if leg == 'Left':
        return 'left_knee_angle_deg'
    elif leg == 'Right':
        return 'right_knee_angle_deg'
    return None


print(f'{datetime.now()} - Functions defined OK')

2026-04-20 21:16:30.809457 - Functions defined OK


-----
## Cell 3 - Select the master frames file and data root folder

Two dialogs will open:
1. Select the **master frames CSV** (`SHQ_SL_Drop_Frames.csv`) — this contains the ground contact frame for each trial.
2. Select the **root Data Collection folder** — the code will look inside each participant subfolder for their drop landing CSV files.

In [4]:
# ── Select master frames file ──────────────────────────────────────────────────
from tkinter.filedialog import askopenfilename

def pick_file(prompt="Select file", filetypes=[("CSV files", "*.csv")]):
    root = Tk()
    root.attributes('-topmost', True)
    root.after(1, root.attributes, '-topmost', False)
    path = askopenfilename(title=prompt, filetypes=filetypes)
    root.destroy()
    return path

print("Select the master frames CSV file...")
frames_path = pick_file(prompt="Select SHQ_SL_Drop_Frames.csv")
frames_df   = pd.read_csv(frames_path)

print(f"Master file loaded: {os.path.basename(frames_path)}")
print(f"  {len(frames_df)} trials across "
      f"{frames_df['ID'].nunique()} participants")
display(frames_df.head(6))

# ── Select root data folder ────────────────────────────────────────────────────
print("\nSelect the root Data Collection folder...")
data_root = pick_directory(
    prompt="Select the root Data Collection folder (contains SHQXXX subfolders)"
)
print(f"Data root: {data_root}")

Select the master frames CSV file...
Master file loaded: SHQ_SL_Drop_Frames.csv
  128 trials across 22 participants


,ID,Trial,Contact_Frame
0,SHQ001,L_drop_1,164
1,SHQ001,L_drop_2,152
2,SHQ001,L_drop_3,349
3,SHQ001,R_drop_1,232
4,SHQ001,R_drop_2,268
5,SHQ001,R_drop_3,225



Select the root Data Collection folder...
Data root: E:/Research/0-Mentoring/MoosbruggerJ/Data Collection


-----
## Cell 4 - Extract knee flexion angles

For each trial in the master file, the code:

1. Navigates to the participant's folder and finds the matching drop landing CSV.
2. Reads the knee angle column for the appropriate leg (left or right).
3. Extracts the **knee flexion angle at initial contact** using the frame index from the master file.
4. Finds the **peak knee flexion angle** in the post-contact window (from contact frame to end of trial).

Any missing files are flagged with a warning and skipped.

In [5]:
results = []
missing = []

for _, row in frames_df.iterrows():
    participant_id = row['ID']
    trial_name     = row['Trial']
    contact_frame  = int(row['Contact_Frame'])
    leg            = parse_leg_from_trial(trial_name)
    knee_col       = get_knee_angle_col(leg)

    if leg is None:
        print(f'⚠  Could not parse leg from trial name: {trial_name} — skipping.')
        missing.append({'ID': participant_id, 'Trial': trial_name,
                        'Reason': 'Could not parse leg from trial name'})
        continue

    # Build expected file path
    # Structure: data_root / SHQXXX / Full Data / trial_name.csv
    trial_file = os.path.join(
        data_root, participant_id, 'Full Data', f'{trial_name}.csv'
    )

    if not os.path.exists(trial_file):
        print(f'⚠  File not found: {trial_file}')
        missing.append({'ID': participant_id, 'Trial': trial_name,
                        'Reason': 'File not found'})
        continue

    # Load the file
    try:
        df = pd.read_csv(trial_file)
    except Exception as e:
        print(f'⚠  Error reading {trial_file}: {e}')
        missing.append({'ID': participant_id, 'Trial': trial_name,
                        'Reason': f'Read error: {e}'})
        continue

    # Validate frame index is within file bounds
    if contact_frame >= len(df):
        print(f'⚠  Contact frame {contact_frame} out of bounds '
              f'(file has {len(df)} rows): {participant_id} {trial_name}')
        missing.append({'ID': participant_id, 'Trial': trial_name,
                        'Reason': f'Contact frame {contact_frame} out of bounds'})
        continue

    # Extract values
    knee_angles = df[knee_col].values
    contact_angle = float(knee_angles[contact_frame])
    post_contact = knee_angles[contact_frame:]
    peak_angle = float(np.nanmax(post_contact))
    peak_frame = contact_frame + int(np.nanargmax(post_contact))
    contact_time = float(df['time_s'].iloc[contact_frame])
    peak_time = float(df['time_s'].iloc[peak_frame])
    time_to_peak = round(peak_time - contact_time, 4)
    knee_excursion = float(peak_angle - contact_angle)

    results.append({
        'ID':                   participant_id,
        'Trial':                trial_name,
        'Leg':                  leg,
        'KneeFlexion_Contact_deg': round(contact_angle, 4),
        'KneeFlexion_Peak_deg':    round(peak_angle,   4),
        'KneeExcursion': round(knee_excursion, 4)
    })

results_df = pd.DataFrame(results)

print(f'\n✓ Done — {len(results)} trials processed, {len(missing)} skipped.')
if missing:
    print('\nSkipped trials:')
    for m in missing:
        print(f"  {m['ID']} {m['Trial']}: {m['Reason']}")

display(results_df)

⚠  File not found: E:/Research/0-Mentoring/MoosbruggerJ/Data Collection\SHQ018\Full Data\R_drop_1.csv
⚠  File not found: E:/Research/0-Mentoring/MoosbruggerJ/Data Collection\SHQ018\Full Data\R_drop_2.csv
⚠  File not found: E:/Research/0-Mentoring/MoosbruggerJ/Data Collection\SHQ018\Full Data\R_drop_3.csv

✓ Done — 125 trials processed, 3 skipped.

Skipped trials:
  SHQ018 R_drop_1: File not found
  SHQ018 R_drop_2: File not found
  SHQ018 R_drop_3: File not found


,ID,Trial,Leg,KneeFlexion_Contact_deg,KneeFlexion_Peak_deg,KneeExcursion
0,SHQ001,L_drop_1,Left,4.9842,59.1526,54.1685
1,SHQ001,L_drop_2,Left,6.3182,61.9315,55.6133
2,SHQ001,L_drop_3,Left,6.6188,61.6006,54.9818
3,SHQ001,R_drop_1,Right,10.0512,63.7296,53.6784
4,SHQ001,R_drop_2,Right,11.1677,76.1796,65.0119
...,...,...,...,...,...,...
120,SHQ022,L_drop_2,Left,12.1296,57.3537,45.2241
121,SHQ022,L_drop_3,Left,9.3994,64.8662,55.4668
122,SHQ022,R_drop_1,Right,27.5207,71.6650,44.1444
123,SHQ022,R_drop_2,Right,14.2495,73.5763,59.3269


-----
## Cell 5 - Quick summary

Group by participant and leg to get mean contact and peak knee flexion — useful sanity check before saving.

In [6]:
if results_df.empty:
    print("No results to summarize — check Cell 4 for errors.")
else:
    summary = (
        results_df
        .groupby(['ID', 'Leg'])
        .agg(
            n_trials = ('Trial', 'count'),
            mean_contact_angle_deg = ('KneeFlexion_Contact_deg', 'mean'),
            mean_peak_angle_deg = ('KneeFlexion_Peak_deg',    'mean'),
            mean_excursion = ('KneeExcursion', 'mean')
        )
        .round(2)
        .reset_index()
    )
    print('Summary — mean values per participant per leg:')
    display(summary)

Summary — mean values per participant per leg:


,ID,Leg,n_trials,mean_contact_angle_deg,mean_peak_angle_deg,mean_excursion
0,SHQ001,Left,3,5.97,60.89,54.92
1,SHQ001,Right,3,10.34,68.65,58.31
2,SHQ002,Left,3,12.28,52.25,39.98
3,SHQ002,Right,3,10.03,52.33,42.30
4,SHQ003,Left,3,11.74,58.20,46.45
5,SHQ003,Right,3,8.72,61.83,53.10
6,SHQ004,Left,3,14.11,59.82,45.71
7,SHQ004,Right,3,8.29,53.77,45.48
8,SHQ005,Left,3,13.07,45.40,32.32
9,SHQ005,Right,3,8.13,42.45,34.32


-----
## Cell 6 - Save results to Excel

A Save As dialog will open. The filename is pre-filled with today's date. If the file already exists at the chosen path, new rows are **appended** and duplicates on `ID + Trial` are dropped (keeping the most recent), so re-running a participant overwrites their previous entry cleanly.

In [8]:
if results_df.empty:
    print("No results to save — complete Cell 4 first.")
else:
    today_str    = datetime.now().strftime('%Y-%m-%d')
    default_name = f"SHQ_DropLanding_results_{today_str}.csv"
    save_path    = pick_save_path(default_name)

    if not save_path:
        print("Save cancelled.")
    else:
        col_order = [
            'ID', 'Trial', 'Leg', 'KneeFlexion_Contact_deg', 'KneeFlexion_Peak_deg', 'KneeExcursion'
        ]
        new_df = results_df[col_order]

        if os.path.exists(save_path):
            existing_df = pd.read_csv(save_path)
            combined_df = pd.concat([existing_df, new_df], ignore_index=True)
            combined_df = combined_df.drop_duplicates(
                subset=['ID', 'Trial'], keep='last'
            )
            print(f"Existing file found — appending. "
                  f"({len(existing_df)} existing rows, "
                  f"{len(new_df)} new rows, duplicates removed.)")
        else:
            combined_df = new_df
            print(f"Creating new file with {len(new_df)} rows.")

        combined_df.to_csv(save_path, index=False)
        print(f"Saved to  : {save_path}")
        print(f"Total rows: {len(combined_df)}")
        display(new_df)

Creating new file with 125 rows.
Saved to  : E:/Research/0-Mentoring/MoosbruggerJ/Data Collection/SHQ_SLDropLanding_results_2026-04-20.xlsx
Total rows: 125


,ID,Trial,Leg,KneeFlexion_Contact_deg,KneeFlexion_Peak_deg,KneeExcursion
0,SHQ001,L_drop_1,Left,4.9842,59.1526,54.1685
1,SHQ001,L_drop_2,Left,6.3182,61.9315,55.6133
2,SHQ001,L_drop_3,Left,6.6188,61.6006,54.9818
3,SHQ001,R_drop_1,Right,10.0512,63.7296,53.6784
4,SHQ001,R_drop_2,Right,11.1677,76.1796,65.0119
...,...,...,...,...,...,...
120,SHQ022,L_drop_2,Left,12.1296,57.3537,45.2241
121,SHQ022,L_drop_3,Left,9.3994,64.8662,55.4668
122,SHQ022,R_drop_1,Right,27.5207,71.6650,44.1444
123,SHQ022,R_drop_2,Right,14.2495,73.5763,59.3269
